In [1]:
import os
import ast
import json 
import getpass
import logging
from pathlib import Path
from time import sleep

from langchain_core.tools import Tool
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import BaseTool

from typing import TypedDict, Dict, List

from PyDI.io import load_xml, load_parquet, load_csv
from PyDI.profiling import DataProfiler

D:\Studium\Master\Semester 5\Team Project\workspace\test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Initialize

In [14]:
OUTPUT_DIR = "output/"
INPUT_DIR = "input/"

# Edit to use different LLM
USE_LLM = "gemini"
#USE_LLM = "groq"

In [16]:
if USE_LLM == "gemini":
    if "GOOGLE_API_KEY" not in os.environ:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")
elif USE_LLM == "groq":
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [17]:
logging.basicConfig(filename= OUTPUT_DIR + 'agent.log',
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%H:%M:%S',
                    level=logging.DEBUG,
                    encoding='utf-8')

## Utilities

In [18]:
def load_dataset(path):
    # check file exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at: {path}")
    ext = os.path.splitext(path)[1].lower()

    # load dataset according to extension
    if ext == ".parquet":
        df = load_parquet(path)
    elif ext == ".csv":
        df = load_csv(path)
    elif ext == ".xml":
        df = load_xml(path, nested_handling="aggregate")
    else:
        raise ValueError(f"Unsupported format: {ext}. Supported: .csv, .parquet, .xml")
    return df
    

## Tools

In [19]:
class ProfileDatasetTool(BaseTool):
    name: str = "profile_dataset"
    description: str = """
        A tool that takes the path as a argument called `path` of type str of the dataset file as string and performs data analysis. 
        A JSON string is returned with the profile data.
    """

    if USE_LLM == "gemini":
        def _run(self, *args, **kwargs) -> str:
            
            # Accept path either positionally or as a kwarg (e.g. "__arg1")
            path = None
            if args:
                # first positional arg
                path = args[0]
            elif kwargs:
                # sometimes the LLM -> function call maps the single arg to "__arg1"
                # or some other autogenerated key. If there's exactly one kwarg, take it.
                if len(kwargs) == 1:
                    path = list(kwargs.values())[0]
                else:
                    # if multiple kwargs provided, try to find a param named "path"
                    path = kwargs.get("path") or kwargs.get("file_path") or None

            return self.create_profile(path)
            
    elif USE_LLM == "groq":
        def _run(self, path) -> str:   
            return self.create_profile(path)

    def create_profile(self, path):

        if not path or not isinstance(path, str):
                raise ValueError("ProfileDatasetTool requires a single path string argument.")

        df = load_dataset(path)    
        
        if df is None or getattr(df, "empty", False):
            # return a structured JSON error string (LLM will see this as content)
            return json.dumps({"error": f"Dataset at {path} loaded as empty or failed to load."})

        profiler = DataProfiler()
        profile = profiler.summary(df, print_summary=False)

        # ensure to return a JSON string (your docstring promised JSON string)
        try:
            profile_json = json.dumps(profile, default=str)
        except Exception:
            # fallback: convert to str if not json-serializable
            profile_json = json.dumps({"profile_str": str(profile)})

        return profile_json
        

## Agents

In [20]:
class SimpleModelAgentState(TypedDict):
    datasets: list
    data_profiles: Dict
    integration_pipeline_code: str

class SimpleModelAgent:
    
    def __init__(self, model, profileTool):
        # initialize logger
        self.logger = logging.getLogger()
        
        # prepare the StateGraph
        graph = StateGraph(SimpleModelAgentState)

        # create nodes
        graph.add_node("profile_data", self.profile_data)
        graph.add_node("pipeline_adaption", self.pipeline_adaption)

        # create edges
        graph.add_edge("profile_data", "pipeline_adaption")
        graph.add_edge("pipeline_adaption", END)

        graph.set_entry_point("profile_data")
        self.graph = graph.compile()

        # make tools available to the model based on model type
        self.tools = {profileTool.name: profileTool}
        if USE_LLM == "groq":
            self.model = model.bind_tools([profileTool])
        elif USE_LLM == "gemini":
            self.model = model.bind_tools(profileTool)
        
    # Creates tool calls to profile the data and saves it into agent state 
    def profile_data(self, state:SimpleModelAgentState):
        self.logger.info('----------------------- Entering profile_data -----------------------')

        system_prompt = """
            You are a data scientist tasked with the integration of several datasets.
            For each dataset path provided, call the tool `profile_dataset` with the path
            (one tool call per dataset).
        """
        
        datasets_list_str = "\n".join(state['datasets'])
        human_content = f"Please profile these datasets (one call per dataset):\n{datasets_list_str}"
        message = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]
        self.logger.info("Input Message:" + str(message))
        
        result = self.model.invoke(message)
        self.logger.info("RESULT:" + str(result))

        # call tools
        tool_calls = result.tool_calls

        self.logger.info("Tool Calls:" + str(tool_calls))
        results = {}
        for t in tool_calls:
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                self.logger.info("adapt_pipeline: ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            
            if USE_LLM == "groq":
                results[t['args']['path']] = result
            elif USE_LLM == "gemini":
                results[t['args']['__arg1']] = result

        with open(OUTPUT_DIR + "profile/profiles.json", "w") as file:
            file.write(json.dumps(results, indent=2))
          
        self.logger.info('Leaving profile_data')
        return {'data_profiles': results}

    def pipeline_adaption(self, state: SimpleModelAgentState):
        self.logger.info('----------------------- Entering pipeline_adaption -----------------------')
    
        # Load example pipeline code
        example_pipeline_path = os.path.join(INPUT_DIR, "example_pipelines/restaurants-goldstandard-integration-pipeline-RB.py")
        if not os.path.exists(example_pipeline_path):
            raise FileNotFoundError(f"Example pipeline not found at: {example_pipeline_path}")
    
        with open(example_pipeline_path, "r", encoding="utf-8") as f:
            example_pipeline_code = f.read()

        # Prepare first-row previews for each dataset
        dataset_previews = {}
        for path in state['datasets']:
            df = load_dataset(path)
            if df is not None and not df.empty:
                # Take only first row and convert to dictionary
                dataset_previews[path] = df.iloc[0].to_dict()
            else:
                dataset_previews[path] = "Failed to load or empty dataset"
                
        # Prepare prompt for the model
        system_prompt = f"""
        You are a data scientist tasked with the integration of several datasets.
        You are provided with the following inputs:
    
        1. An example integration pipeline (Python code using PyDI library):
        {example_pipeline_code}
    
        2. A list of dataset file locations:
        {json.dumps(state['datasets'], indent=2)}
    
        3. The first row of each dataset to help understand the structure:
        {json.dumps(dataset_previews, indent=2)}
    
        4. A dictionary containing the profile data of the datasets
        (including nulls_per_column and dtypes of the columns):
        {json.dumps(state['data_profiles'], indent=2)}
    
        Your task is to **create a similar integration pipeline** so that it works with
        the datasets provided. Output should only consist of the relevant Python code
        for the integration pipeline.
        """

        human_content = "Please adjust the example integration pipeline accordingly."
    
        message = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_content)
        ]
    
        self.logger.info("Input Message:" + str(message))
    
        # Call the model
        adapted_pipeline = self.model.invoke(message)
        adapted_pipeline = adapted_pipeline.content if hasattr(adapted_pipeline, "content") else str(adapted_pipeline)

        self.logger.info("RESULT:" + str(adapted_pipeline))
        
        # Strip leading/trailing ``` if present
        #if adapted_pipeline.startswith("```") and adapted_pipeline.endswith("```"):
        #    adapted_pipeline = adapted_pipeline[3:-3].strip()
        
        with open(OUTPUT_DIR + "pipeline_code/pipeline.py", 'w') as file:
            file.write(str(adapted_pipeline))
    
        self.logger.info('Leaving pipeline_adaption')
        return {'integration_pipeline_code': adapted_pipeline}


## Invoke Pipeline

In [21]:
# Initialize model
if USE_LLM == "gemini":
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
elif USE_LLM == "groq":
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )

# input datasets
#datasets = [
#    INPUT_DIR + "datasets/kaggle_small.parquet",
#    INPUT_DIR + "datasets/uber_eats_small.parquet",
#    INPUT_DIR + "datasets/yelp_small.parquet"
#]

datasets = [
    INPUT_DIR + "datasets/amazon_small.parquet",
    INPUT_DIR + "datasets/goodreads_small.parquet",
    INPUT_DIR + "datasets/metabooks_small.parquet"
]

# prepare agent
agent = SimpleModelAgent(llm, ProfileDatasetTool())


# invoke agent
result = agent.graph.invoke({"datasets": datasets})


In [13]:
print(result['integration_pipeline_code'])

```python
from PyDI.io import load_parquet
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher

from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
from PyDI.fusion import DataFusionEvaluator, tokenized_match

import pandas as pd
import numpy as np

# --------------------------------
# Prepare Data
# Important: Use the correct loader (load_parquet, load_csv, load_xml)
# --------------------------------

# Define dataset paths
DATA_DIR = "input/datasets/"

# Load the first dataset
amazon_dataset = load_parquet(
    DATA_DIR + "amazon_small.parquet",
    name="amazon",
)

# Load Goodreads dataset  
goodreads_dataset = load_parquet(
    DATA_DIR + "goodreads_small.parquet",
    name="goodreads",
)

# Load Metabooks dataset
metabooks_dataset = load_parquet(
    DATA_DIR + "metabooks_small.parquet",
    name="metabooks",
)

# c